In [1]:
%pip install -U langchain langchain-huggingface huggingface-hub sentence-transformers chromadb langchain-chroma langchain-community pypdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

d:\Computer Courses\Agenetic-Ai\langchain_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8583.10it/s]


In [3]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.3
)

model = ChatHuggingFace(llm=llm)

In [5]:
from langchain_core.documents import Document

# Create Langchain documents for IPL Players

doc1 = Document(
    page_content="Virat Kohli is an Indian cricketer and former captain of the Indian national team. He is widely regarded as one of the best batsmen in the world.",
    metadata={"team":"Royal Challengers Bangalore"}
)
doc2 = Document(
    page_content="Rohit Sharma is an Indian cricketer and the current captain of the Indian national team. He is known for his aggressive batting style and has scored multiple double centuries in One Day Internationals.",
    metadata={"team":"Mumbai Indians"}
)
doc3 = Document(
    page_content="MS Dhoni is a former Indian cricketer and captain of the Indian national team. He is known for his calm demeanor and exceptional leadership skills, leading India to numerous victories in international cricket.",
    metadata={"team":"Chennai Super Kings"}
)
doc4 = Document(
    page_content="Jasprit Bumrah is an Indian cricketer and one of the best fast bowlers in the world. He is known for his unique bowling action and ability to bowl yorkers consistently.",
    metadata={"team":"Mumbai Indians"}
)  
doc5 = Document(
    page_content="Ravindra Jadeja is an Indian cricketer and an all-rounder. He is known for his exceptional fielding skills, accurate left-arm spin bowling, and ability to contribute with the bat.",
    metadata={"team":"Chennai Super Kings"}
)


In [6]:
from langchain_chroma import Chroma
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory='chroma_db',
    collection_name='sample'
)

In [7]:
docs = [doc1, doc2, doc3, doc4, doc5]
# add documents
vector_store.add_documents(docs)

['db311377-12b3-49b8-8cd5-201dbc63c765',
 '0edf4827-bdcf-4418-8699-ffafe02cf63e',
 '3d0ee432-689d-43af-a385-3c7810d54011',
 '47566095-165c-4ea8-9cdc-e451b1c9afc5',
 '568b8b94-1e9d-4120-899b-10e4dbc4e227']

In [8]:
# view documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['3d565778-68e3-40f7-b271-95f64ef53bc6',
  'a9d24348-4320-4fae-b1aa-407eefd0350d',
  'c9081bc1-6bbd-4d3f-b53e-a8e792dc2cb7',
  '73a24773-b1da-422a-8405-6e7c7d7c5963',
  'fef31ca2-007e-4983-96aa-a1aa079db714',
  '1b6a7a3b-b561-4bfa-abd0-a0afd8099468',
  '203ced0f-4b33-4f2d-9a00-564dff0ce85a',
  '3b6b409a-c1f3-4594-ba7a-f80a352c6645',
  'a4bccd1b-c455-4ae1-afc5-d2d9b5a9e314',
  'db311377-12b3-49b8-8cd5-201dbc63c765',
  '0edf4827-bdcf-4418-8699-ffafe02cf63e',
  '3d0ee432-689d-43af-a385-3c7810d54011',
  '47566095-165c-4ea8-9cdc-e451b1c9afc5',
  '568b8b94-1e9d-4120-899b-10e4dbc4e227'],
 'embeddings': array([[ 0.01533243,  0.0745532 , -0.04349047, ...,  0.01531935,
          0.05182592, -0.01778253],
        [ 0.01290081,  0.01218231, -0.03368595, ..., -0.0092599 ,
         -0.00696487, -0.02739695],
        [-0.02228261,  0.05256926,  0.0630175 , ...,  0.02790222,
          0.00276882, -0.06323092],
        ...,
        [-0.02228261,  0.05256926,  0.0630175 , ...,  0.02790222,
     

In [9]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='73a24773-b1da-422a-8405-6e7c7d7c5963', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is an Indian cricketer and one of the best fast bowlers in the world. He is known for his unique bowling action and ability to bowl yorkers consistently.'),
 Document(id='3b6b409a-c1f3-4594-ba7a-f80a352c6645', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is an Indian cricketer and one of the best fast bowlers in the world. He is known for his unique bowling action and ability to bowl yorkers consistently.')]

In [10]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='73a24773-b1da-422a-8405-6e7c7d7c5963', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is an Indian cricketer and one of the best fast bowlers in the world. He is known for his unique bowling action and ability to bowl yorkers consistently.'),
  0.945003867149353),
 (Document(id='3b6b409a-c1f3-4594-ba7a-f80a352c6645', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is an Indian cricketer and one of the best fast bowlers in the world. He is known for his unique bowling action and ability to bowl yorkers consistently.'),
  0.945003867149353)]

In [11]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query='',
    filter={"team":"Chennai Super Kings"}
)

[(Document(id='c9081bc1-6bbd-4d3f-b53e-a8e792dc2cb7', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni is a former Indian cricketer and captain of the Indian national team. He is known for his calm demeanor and exceptional leadership skills, leading India to numerous victories in international cricket.'),
  1.7571805715560913),
 (Document(id='203ced0f-4b33-4f2d-9a00-564dff0ce85a', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni is a former Indian cricketer and captain of the Indian national team. He is known for his calm demeanor and exceptional leadership skills, leading India to numerous victories in international cricket.'),
  1.7571805715560913),
 (Document(id='3d0ee432-689d-43af-a385-3c7810d54011', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni is a former Indian cricketer and captain of the Indian national team. He is known for his calm demeanor and exceptional leadership skills, leading India to numerous victories in internation

In [12]:
# Update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captian of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistency.",
    metadata={"team":"Royal Challengers Bangalore"}
)

vector_store.update_documents(ids=['db311377-12b3-49b8-8cd5-201dbc63c765'],documents=[updated_doc1])

In [13]:
# view documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['3d565778-68e3-40f7-b271-95f64ef53bc6',
  'a9d24348-4320-4fae-b1aa-407eefd0350d',
  'c9081bc1-6bbd-4d3f-b53e-a8e792dc2cb7',
  '73a24773-b1da-422a-8405-6e7c7d7c5963',
  'fef31ca2-007e-4983-96aa-a1aa079db714',
  '1b6a7a3b-b561-4bfa-abd0-a0afd8099468',
  '203ced0f-4b33-4f2d-9a00-564dff0ce85a',
  '3b6b409a-c1f3-4594-ba7a-f80a352c6645',
  'a4bccd1b-c455-4ae1-afc5-d2d9b5a9e314',
  'db311377-12b3-49b8-8cd5-201dbc63c765',
  '0edf4827-bdcf-4418-8699-ffafe02cf63e',
  '3d0ee432-689d-43af-a385-3c7810d54011',
  '47566095-165c-4ea8-9cdc-e451b1c9afc5',
  '568b8b94-1e9d-4120-899b-10e4dbc4e227'],
 'embeddings': array([[ 0.01533243,  0.0745532 , -0.04349047, ...,  0.01531935,
          0.05182592, -0.01778253],
        [ 0.01290081,  0.01218231, -0.03368595, ..., -0.0092599 ,
         -0.00696487, -0.02739695],
        [-0.02228261,  0.05256926,  0.0630175 , ...,  0.02790222,
          0.00276882, -0.06323092],
        ...,
        [-0.02228261,  0.05256926,  0.0630175 , ...,  0.02790222,
     

In [14]:
# delete document
vector_store.delete(ids=['db311377-12b3-49b8-8cd5-201dbc63c765'])

In [15]:
# view documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['3d565778-68e3-40f7-b271-95f64ef53bc6',
  'a9d24348-4320-4fae-b1aa-407eefd0350d',
  'c9081bc1-6bbd-4d3f-b53e-a8e792dc2cb7',
  '73a24773-b1da-422a-8405-6e7c7d7c5963',
  'fef31ca2-007e-4983-96aa-a1aa079db714',
  '1b6a7a3b-b561-4bfa-abd0-a0afd8099468',
  '203ced0f-4b33-4f2d-9a00-564dff0ce85a',
  '3b6b409a-c1f3-4594-ba7a-f80a352c6645',
  'a4bccd1b-c455-4ae1-afc5-d2d9b5a9e314',
  '0edf4827-bdcf-4418-8699-ffafe02cf63e',
  '3d0ee432-689d-43af-a385-3c7810d54011',
  '47566095-165c-4ea8-9cdc-e451b1c9afc5',
  '568b8b94-1e9d-4120-899b-10e4dbc4e227'],
 'embeddings': array([[ 0.01533243,  0.0745532 , -0.04349047, ...,  0.01531935,
          0.05182592, -0.01778253],
        [ 0.01290081,  0.01218231, -0.03368595, ..., -0.0092599 ,
         -0.00696487, -0.02739695],
        [-0.02228261,  0.05256926,  0.0630175 , ...,  0.02790222,
          0.00276882, -0.06323092],
        ...,
        [-0.02228261,  0.05256926,  0.0630175 , ...,  0.02790222,
          0.00276882, -0.06323092],
        [ 0